# Business qnalysis with pandas groupby

This notebook analyses the enriched transaction dataset at customer, segment, merchant, category and monthly levels.

## Objectives

- Use groupby() to divide data into groups
- Calculate basic grouped statistics
- Create multiple named aggregations
- Group by one or several columns
- Calculate customer, merchant and category indicators
- Create monthly activity summaries
- Prepare customer level metrics for later anomaly analysis

In [20]:
from pathlib import Path
import pandas as pd

In [21]:
current_directory = Path.cwd()

if current_directory.name == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory

project_root

WindowsPath('c:/Users/Focus/Documents/GitHub/transaction-intelligence')

In [22]:
enriched_transactions_path = (
    project_root
    / "data"
    / "processed"
    / "transactions_enriched.csv"
)

enriched_transactions_path.exists()

True

## Loading enriched transactions

The enriched dataset contains transaction, account, customer and merchant information in a single table

In [23]:
transactions = pd.read_csv(
    enriched_transactions_path,
    parse_dates=[
        "timestamp",
        "opening_date",
        "signup_date",
    ],
)

transactions.head()

,transaction_id,account_id,merchant_id,timestamp,amount,currency,transaction_type,status,year,month,...,customer_country,signup_date,customer_segment,merchant_name,merchant_category,merchant_country,online,is_international,transaction_scope,currency_matches_account
0,T0013,A002,M013,2025-01-18 22:17:00,54.99,EUR,Purchase,Completed,2025,1,...,Spain,2023-01-15,Standard,Game Box,Entertainment,France,True,True,International,True
1,T0017,A001,M002,2025-01-24 17:18:00,6.80,EUR,Purchase,Completed,2025,1,...,Spain,2023-01-15,Standard,Urban Coffee,Restaurants,Spain,False,False,Domestic,True
2,T0032,A001,M010,2025-02-14 21:10:00,15.50,EUR,Purchase,Completed,2025,2,...,Spain,2023-01-15,Standard,Cinema Central,Entertainment,Spain,False,False,Domestic,True
3,T0001,A001,M001,2025-03-01 08:15:00,34.80,EUR,Purchase,Completed,2025,3,...,Spain,2023-01-15,Standard,Green Market,Groceries,Spain,False,False,Domestic,True
4,T0043,A002,M004,2025-03-03 19:55:00,899.00,EUR,Purchase,Completed,2025,3,...,Spain,2023-01-15,Standard,Tech World,Electronics,Germany,True,True,International,True


In [24]:
transactions.shape

(52, 40)

In [25]:
# Before grouping, I create aux bools for future calculus simplification

transactions["is_completed"] = (transactions["status"].eq("Completed"))
transactions["is_declined"] = (transactions["status"].eq("Declined"))
transactions["is_refund"] = (transactions["status"].eq("Refund"))

In [26]:
transactions[
    [
        "transaction_id",
        "status",
        "is_completed",
        "is_declined",
        "transaction_type",
        "is_refund",
    ]
].head()

,transaction_id,status,is_completed,is_declined,transaction_type,is_refund
0,T0013,Completed,True,False,Purchase,False
1,T0017,Completed,True,False,Purchase,False
2,T0032,Completed,True,False,Purchase,False
3,T0001,Completed,True,False,Purchase,False
4,T0043,Completed,True,False,Purchase,False


# Customer level analysis

I begin by grouping transactions by customer

In [27]:
transactions_per_customer = (transactions.groupby("customer_id").size())
transactions_per_customer

customer_id
C001    8
C002    6
C003    4
C004    6
C005    4
C006    4
C007    3
C008    4
C009    3
C010    3
C011    4
C012    3
dtype: int64

In [28]:
transactions_per_customer = (transactions.groupby("customer_id").size().reset_index(name = "transaction_count"))
transactions_per_customer.head()

,customer_id,transaction_count
0,C001,8
1,C002,6
2,C003,4
3,C004,6
4,C005,4


In [29]:
customer_size = (transactions.groupby("customer_id").size()) # rows
customer_count = (transactions.groupby("customer_id")["transaction_id"].count()) # notna/col

In [30]:
customer_size.equals(customer_count)

True

In [31]:
# Net movement per client

customer_net_amount = (transactions.groupby("customer_id")["net_amount"].sum().sort_values(ascending=False))
customer_net_amount

customer_id
C002    1255.79
C008    1162.25
C001    1087.09
C004     815.40
C009     289.49
C005     214.50
C011     186.10
C010     177.99
C006     170.85
C003     135.97
C007     101.80
C012      73.00
Name: net_amount, dtype: float64

In [32]:
# Mean and median

customer_average_amount = (transactions.groupby("customer_id")["absolute_amount"].mean())
customer_median_amount = (transactions.groupby("customer_id")["absolute_amount"].median())

In [33]:
pd.DataFrame(
    {
        "average_amount": (customer_average_amount),
        "median_amount": (customer_median_amount),
    }
)

,average_amount,median_amount
customer_id,,
C001,140.883750,27.395
C002,209.298333,174.500
C003,33.992500,32.490
C004,212.400000,166.200
C005,81.125000,86.200
C006,49.487500,40.925
C007,33.933333,7.400
C008,290.562500,232.950
C009,96.496667,74.500


In [34]:
# Summary per client with agg()

customer_summary = (
    transactions.groupby(
        ["customer_id", "customer_segment", "customer_country"]
    )
    .agg(
        transactions_count = ("transaction_id", "count"),
        completed_transactions = ("is_completed", "sum"),
        declined_transactions = ("is_declined", "sum"),
        refund_transactions = ("is_refund", "sum"),
        total_absolute_amount = ("absolute_amount", "sum"),
        total_net_amount = ("net_amount", "sum"),
        average_amount=("absolute_amount","mean"),
        median_amount=("absolute_amount","median"),
        maximum_amount=("absolute_amount","max"),
        unique_merchants = ("merchant_id", "nunique"),
        unique_categories = ("merchant_category", "nunique"),
        international_transactions = ("is_international", "sum"),
        total_commission = ("commission_amount", "sum")
    ).reset_index()
)

customer_summary.head()

,customer_id,customer_segment,customer_country,transactions_count,completed_transactions,declined_transactions,refund_transactions,total_absolute_amount,total_net_amount,average_amount,median_amount,maximum_amount,unique_merchants,unique_categories,international_transactions,total_commission
0,C001,Standard,Spain,8,8,0,0,1127.07,1087.09,140.883750,27.395,899.0,7,6,3,16.59
1,C002,Premium,Spain,6,6,0,0,1255.79,1255.79,209.298333,174.500,540.0,6,4,4,18.83
2,C003,Student,France,4,4,0,0,135.97,135.97,33.992500,32.490,52.0,3,3,4,2.03
3,C004,Premium,Germany,6,5,1,0,1274.40,815.40,212.400000,166.200,459.0,5,5,4,12.23
4,C005,Standard,Spain,4,3,1,0,324.50,214.50,81.125000,86.200,130.6,4,4,3,3.22


In [35]:
customer_summary[
    [
        "total_absolute_amount",
        "total_net_amount",
        "average_amount",
        "median_amount",
        "maximum_amount",
        "total_commission",
    ]
] = (
    customer_summary[
        [
            "total_absolute_amount",
            "total_net_amount",
            "average_amount",
            "median_amount",
            "maximum_amount",
            "total_commission",
        ]
    ]
    .round(2)
)

In [36]:
# % of international transactions

customer_summary["international_transaction_rate"] = (
    customer_summary["international_transactions"]
    .div(
        customer_summary["transactions_count"]
    )
    .mul(100)
    .round(2)
)

In [37]:
# % declined transactions
customer_summary[
    "decline_rate"
] = (
    customer_summary[
        "declined_transactions"
    ]
    .div(
        customer_summary[
            "transactions_count"
        ]
    )
    .mul(100)
    .round(2)
)

In [38]:
# I will sort clients looking at their net movement

customers_summary = (customers_summary
    .sort_values(
        "total_net_amount",
        ascending = False
    )
    .reset_index(drop = True))

customers_summary.head()

NameError: name 'customers_summary' is not defined

# Customer segment analysis

Customer level transactions are aggregated by customer segment

In [ ]:
segment_summary = (
    transactions
    .groupby("customer_segment")
    .agg(
        unique_customers=(
            "customer_id",
            "nunique",
        ),
        transaction_count=(
            "transaction_id",
            "count",
        ),
        completed_transactions=(
            "is_completed",
            "sum",
        ),
        declined_transactions=(
            "is_declined",
            "sum",
        ),
        total_net_amount=(
            "net_amount",
            "sum",
        ),
        average_transaction_amount=(
            "absolute_amount",
            "mean",
        ),
        total_commission=(
            "commission_amount",
            "sum",
        ),
    )
    .reset_index()
)

In [ ]:
segment_summary[
    [
        "total_net_amount",
        "average_transaction_amount",
        "total_commission",
    ]
] = (
    segment_summary[
        [
            "total_net_amount",
            "average_transaction_amount",
            "total_commission",
        ]
    ]
    .round(2)
)

In [ ]:
segment_summary["decline_rate"] = (
    segment_summary[
        "declined_transactions"
    ]
    .div(
        segment_summary[
            "transaction_count"
        ]
    )
    .mul(100)
    .round(2)
)

In [ ]:
segment_summary.sort_values(
    "total_net_amount",
    ascending=False,
)

,customer_segment,unique_customers,transaction_count,completed_transactions,declined_transactions,total_net_amount,average_transaction_amount,total_commission,decline_rate
1,Standard,5,23,22,1,2751.74,126.16,41.57,4.35
0,Premium,5,22,20,2,2493.03,135.42,37.39,9.09
2,Student,2,7,7,0,425.46,60.78,6.37,0.00


# Merchant category analysis

Transactions are grouped by merchant category to compare activity, customers and financial volume

In [ ]:
category_summary = (
    transactions
    .groupby("merchant_category")
    .agg(
        transaction_count=(
            "transaction_id",
            "count",
        ),
        unique_customers=(
            "customer_id",
            "nunique",
        ),
        unique_merchants=(
            "merchant_id",
            "nunique",
        ),
        total_absolute_amount=(
            "absolute_amount",
            "sum",
        ),
        total_net_amount=(
            "net_amount",
            "sum",
        ),
        average_amount=(
            "absolute_amount",
            "mean",
        ),
        median_amount=(
            "absolute_amount",
            "median",
        ),
        international_transactions=(
            "is_international",
            "sum",
        ),
        total_commission=(
            "commission_amount",
            "sum",
        ),
    )
    .reset_index()
)

In [ ]:
category_summary[
    [
        "total_absolute_amount",
        "total_net_amount",
        "average_amount",
        "median_amount",
        "total_commission",
    ]
] = (
    category_summary[
        [
            "total_absolute_amount",
            "total_net_amount",
            "average_amount",
            "median_amount",
            "total_commission",
        ]
    ]
    .round(2)
)

In [ ]:
category_summary[
    [
        "total_absolute_amount",
        "total_net_amount",
        "average_amount",
        "median_amount",
        "total_commission",
    ]
] = (
    category_summary[
        [
            "total_absolute_amount",
            "total_net_amount",
            "average_amount",
            "median_amount",
            "total_commission",
        ]
    ]
    .round(2)
)

In [ ]:
category_scope_summary = (
    transactions
    .groupby(
        [
            "merchant_category",
            "transaction_scope",
        ]
    )
    .agg(
        transaction_count=(
            "transaction_id",
            "count",
        ),
        total_net_amount=(
            "net_amount",
            "sum",
        ),
        average_amount=(
            "absolute_amount",
            "mean",
        ),
    )
    .reset_index()
)

category_scope_summary[
    [
        "total_net_amount",
        "average_amount",
    ]
] = (
    category_scope_summary[
        [
            "total_net_amount",
            "average_amount",
        ]
    ]
    .round(2)
)

category_scope_summary.head(15)

,merchant_category,transaction_scope,transaction_count,total_net_amount,average_amount
0,Books,Domestic,3,21.50,20.49
1,Books,International,2,43.98,21.99
2,Electronics,Domestic,2,249.90,354.45
3,Electronics,International,6,1451.98,260.33
4,Entertainment,Domestic,2,79.50,39.75
5,Entertainment,International,4,172.97,43.24
6,Fashion,International,5,527.45,105.49
7,Groceries,Domestic,3,106.25,35.42
8,Groceries,International,5,215.60,48.54
9,Health,Domestic,1,17.90,17.90


In [ ]:
# Merchant analysis

merchant_summary = (
    transactions
    .groupby(
        [
            "merchant_id",
            "merchant_name",
            "merchant_category",
            "merchant_country",
        ]
    )
    .agg(
        transaction_count=(
            "transaction_id",
            "count",
        ),
        unique_customers=(
            "customer_id",
            "nunique",
        ),
        total_net_amount=(
            "net_amount",
            "sum",
        ),
        average_amount=(
            "absolute_amount",
            "mean",
        ),
        maximum_amount=(
            "absolute_amount",
            "max",
        ),
        declined_transactions=(
            "is_declined",
            "sum",
        ),
        total_commission=(
            "commission_amount",
            "sum",
        ),
    )
    .reset_index()
)

In [ ]:
merchant_summary[
    [
        "total_net_amount",
        "average_amount",
        "maximum_amount",
        "total_commission",
    ]
] = (
    merchant_summary[
        [
            "total_net_amount",
            "average_amount",
            "maximum_amount",
            "total_commission",
        ]
    ]
    .round(2)
)

merchant_summary = (
    merchant_summary
    .sort_values(
        "total_net_amount",
        ascending=False,
    )
    .reset_index(drop=True)
)

merchant_summary.head(10)

,merchant_id,merchant_name,merchant_category,merchant_country,transaction_count,unique_customers,total_net_amount,average_amount,maximum_amount,declined_transactions,total_commission
0,M015,Sun Hotel,Travel,Portugal,3,3,1575.00,525.00,625.00,0,23.63
1,M004,Tech World,Electronics,Germany,4,3,1498.89,489.47,899.00,1,22.48
2,M008,Travel Now,Travel,Netherlands,3,3,750.00,250.00,320.00,0,11.24
3,M011,Style Avenue,Fashion,Italy,5,5,527.45,105.49,145.90,0,7.91
4,M009,Home Corner,Home,Portugal,3,3,263.85,87.95,130.60,0,3.96
5,M013,Game Box,Entertainment,France,4,4,223.97,55.99,64.00,0,3.35
6,M006,Cloud Store,Electronics,Ireland,4,4,202.99,78.25,110.00,1,3.05
7,M001,Green Market,Groceries,Spain,6,3,188.10,35.87,48.25,1,2.82
8,M007,Fit Zone,Sports,Spain,2,2,134.50,67.25,82.50,0,2.02
9,M005,Fresh Basket,Groceries,France,2,2,133.75,66.88,71.35,0,2.01


# Monthly activity analysis

Transactions are aggregated by year and month

In [ ]:
monthly_summary = (
    transactions
    .groupby(
        [
            "year",
            "month",
            "month_name",
        ]
    )
    .agg(
        transaction_count=(
            "transaction_id",
            "count",
        ),
        unique_customers=(
            "customer_id",
            "nunique",
        ),
        completed_transactions=(
            "is_completed",
            "sum",
        ),
        declined_transactions=(
            "is_declined",
            "sum",
        ),
        total_net_amount=(
            "net_amount",
            "sum",
        ),
        average_amount=(
            "absolute_amount",
            "mean",
        ),
        total_commission=(
            "commission_amount",
            "sum",
        ),
    )
    .reset_index()
    .sort_values(
        [
            "year",
            "month",
        ]
    )
    .reset_index(drop=True)
)

In [ ]:
monthly_summary[
    [
        "total_net_amount",
        "average_amount",
        "total_commission",
    ]
] = (
    monthly_summary[
        [
            "total_net_amount",
            "average_amount",
            "total_commission",
        ]
    ]
    .round(2)
)

monthly_summary

,year,month,month_name,transaction_count,unique_customers,completed_transactions,declined_transactions,total_net_amount,average_amount,total_commission
0,2025,1,January,11,8,10,1,745.63,70.25,11.18
1,2025,2,February,11,10,10,1,895.63,91.42,13.42
2,2025,3,March,12,8,11,1,1985.35,207.03,30.08
3,2025,4,April,2,2,2,0,159.20,79.60,2.38
4,2025,5,May,3,3,3,0,413.99,138.00,6.20
5,2025,6,June,2,2,2,0,267.80,133.90,4.02
6,2025,7,July,2,2,2,0,78.25,39.12,1.17
7,2025,8,August,1,1,1,0,62.40,62.40,0.94
8,2025,9,September,2,2,2,0,108.99,54.50,1.64
9,2025,10,October,2,1,2,0,423.99,212.00,6.36


In [ ]:
# Calculation of general indicators
overall_summary = pd.Series(
    {
        "total_transactions": len(
            transactions
        ),
        "unique_customers": (
            transactions[
                "customer_id"
            ].nunique()
        ),
        "unique_merchants": (
            transactions[
                "merchant_id"
            ].nunique()
        ),
        "completed_transactions": int(
            transactions[
                "is_completed"
            ].sum()
        ),
        "declined_transactions": int(
            transactions[
                "is_declined"
            ].sum()
        ),
        "international_transactions": int(
            transactions[
                "is_international"
            ].sum()
        ),
        "total_absolute_amount": round(
            transactions[
                "absolute_amount"
            ].sum(),
            2,
        ),
        "total_net_amount": round(
            transactions[
                "net_amount"
            ].sum(),
            2,
        ),
        "total_commission": round(
            transactions[
                "commission_amount"
            ].sum(),
            2,
        ),
    },
    name="value",
)

overall_summary

total_transactions              52.00
unique_customers                12.00
unique_merchants                15.00
completed_transactions          49.00
declined_transactions            3.00
international_transactions      37.00
total_absolute_amount         6306.31
total_net_amount              5670.23
total_commission                85.33
Name: value, dtype: float64

In [ ]:
overall_decline_rate = (
    transactions[
        "is_declined"
    ].mean()
    * 100
)

overall_international_rate = (
    transactions[
        "is_international"
    ].mean()
    * 100
)

print(
    "Decline rate:",
    round(overall_decline_rate, 2),
    "%",
)

print(
    "International rate:",
    round(
        overall_international_rate,
        2,
    ),
    "%",
)

Decline rate: 5.77 %
International rate: 71.15 %


In [ ]:
# I will add the average of each customer to each transaction with transform()

transactions["customer_average_amount"] = (transactions.groupby("customer_id")["absolute_amount"].transform("mean"))

In [ ]:
transactions[
    [
        "transaction_id",
        "customer_id",
        "absolute_amount",
        "customer_average_amount",
    ]
].head(15)

,transaction_id,customer_id,absolute_amount,customer_average_amount
0,T0013,C001,54.99,140.883750
1,T0017,C001,6.80,140.883750
2,T0032,C001,15.50,140.883750
3,T0001,C001,34.80,140.883750
4,T0043,C001,899.00,140.883750
5,T0058,C001,76.00,140.883750
6,T0060,C001,19.99,140.883750
7,T0047,C001,19.99,140.883750
8,T0014,C002,12.60,209.298333
9,T0033,C002,540.00,209.298333


In [ ]:
transactions["customer_total_absolute_amount"] = (transactions.groupby("customer_id")["absolute_amount"].transform("sum"))
transactions["customer_transaction_count"] = (transactions.groupby("customer_id")["transaction_id"].transform("count"))


In [ ]:
transactions[
    [
        "transaction_id",
        "customer_id",
        "absolute_amount",
        "customer_average_amount",
        "customer_total_absolute_amount",
        "customers_transaction_count",
    ]
].head(15)

KeyError: "['customers_transaction_count'] not in index"